# Lecture 02 ADVANCED homework: train on the course glossary, compare with glossary search

This is the exercise notebook adapted for the **Advanced** homework. What changed versus the original:

| Cell | Change |
| --- | --- |
| Settings | `num_train_epochs` 8, `warmup_steps` 10, `max_new_tokens` 160 (glossary definitions are up to ~60 words and were cut off at 96 tokens) |
| Section 2 | Data comes from `scripts/make_sft_dataset.py` (JSONL) wrapped in the same `{"answer", "needs_review"}` JSON, plus ungrounded examples; new 42-question test set with 4 kinds + ungrounded |
| Section 3 | `SYSTEM` / `build_messages`: **closed-book** (no FAQ text in the prompt, the model must recall the glossary from its weights) |
| Section 5b | New: per-kind comparison and a **glossary search** baseline (no training) |
| Section 6 / 7 | Knowledge questions and `result.json` adapted |

Before running: upload `data/glossary.json` and `glossary_sft.jsonl` (created in Codespaces with `uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl`) using the upload cell below. Text in the original cells that talks about the FAQ refers to the original exercise.


## 0. Check the Environment and Install Libraries

**What to Do**: Check that a GPU is allocated and install libraries pinned to verified versions.

**What to Look For**

- The first cell displays `Tesla T4`.
- The second cell ends with a list of versions (such as `transformers: 5.18.0`). Installation takes about 1 minute.

**Troubleshooting**

- "No GPU found": select **Runtime → Change runtime type → T4 GPU** and run the cell again. If demand prevents allocation, try again later.
- Red pip `ERROR: pip's dependency resolver ...` messages (about `cudf` or `pyarrow`) warn of conflicts with other packages preinstalled in Colab. They do not affect this exercise, so continue.

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU runtimes do not have nvidia-smi
    gpu_info = ""
print(gpu_info or "No GPU found. Select Runtime → Change runtime type → T4 GPU.")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Versions verified on a Colab T4 (Python 3.13, torch 2.11.0+cu130). Use the torch bundled with Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 requires transformers 5.x
    "peft": "0.21.1",           # LoRA implementation
    "trl": "1.14.2",            # Training loop (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # The bundled torchao (0.10) is incompatible with peft 0.21 and causes ImportError during training. Remove it since it is unused
%pip install -q {spec}

# Dependency warnings about cudf-cu13 and pyarrow appear as ERROR, but can be ignored because cudf is unused
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.14.1+cu130'}


## 1. Settings

**What to Do**: Collect the model name, number of training epochs (passes through the data), and other settings in a single dictionary, `CONFIG`. This dictionary is recorded as is in the submission file in section 7 so you can later identify "the settings used for each experiment".

- Fixing `revision` ensures the same weights are used whenever you run the notebook. This keeps conditions consistent for before/after and cross-person comparisons.
- `num_train_epochs: 20` means 20 passes through the training data (14 examples). With 2 steps per pass, this totals 40 steps. The original Japanese run took 4–5 minutes on a T4; timing may differ with the translated data.
- `gen: {"do_sample": False}` uses greedy generation. This fixes generation conditions before and after training and removes the effect of sampling variation.

Setting `USE_DRIVE` to `True` mounts Google Drive and saves the trained adapter and result files to Drive as well. Use this to retain results if the session disconnects. Leave it `False` if you will finish in one session.

**What to Look For**: The settings are displayed as JSON.

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # Small instruction-tuned model (Apache-2.0). Fits on a T4 without quantization
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # Pinned to use the same weights on every run
    "seed": 42,
    "num_train_epochs": 8,   # Homework: ~148 training examples -> ~19 steps per epoch -> ~152 steps. Lower it if time is short
    "warmup_steps": 10,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA matrix dimension (smaller means fewer trainable parameters)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 160,   # glossary definitions are longer than FAQ answers; 96 would cut the JSON off
    "gen": {"do_sample": False},   # Fix evaluation to greedy generation (consistent before/after conditions)
    "work_dir": "/content/qlora_lab",                          # Result file location (visible in the file list on the left)
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # Save location when USE_DRIVE=True
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # Set True to mount Google Drive and save adapters and outputs
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 8,
  "warmup_steps": 10,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 160,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


In [4]:
# Upload data/glossary.json and glossary_sft.jsonl (Codespaces: uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl)
from google.colab import files
_up = files.upload()          # files land in /content, which is where the data cell expects them
print(list(_up))


Saving glossary.json to glossary (1).json
Saving glossary_sft.jsonl to glossary_sft (1).jsonl
['glossary (1).json', 'glossary_sft (1).jsonl']


## 2. Glossary training data and a new test set

**What to Do**: Build train / valid / test from the course glossary (65 terms).

- **Training data**: the 130 Q&A pairs from `make_sft_dataset.py` (2 question templates per term), with each definition wrapped as `{"answer": <definition>, "needs_review": false}`. Plus 24 **ungrounded** examples (12 terms that are not in the glossary, 2 templates each) answered with "This glossary does not contain that information..." and `needs_review: true`.
- **Validation**: 6 held-out "Briefly explain X." questions and 2 ungrounded ones (loss monitoring only).
- **Test (42 questions)**: `direct` (12, same wording as training), `rephrase` (12, new wording), `alias` (6, e.g. "What is HITL?"), `descriptive` (4, the term is never named), `ungrounded` (8, terms absent from the glossary and from training).
- `gold`: `needs_review` plus one required phrase copied from the definition (the cell asserts that it really occurs in the definition).

**What to Look For**: `train=148 valid=8 test=42 gold=42` and the per-kind counts.


In [5]:
# ---- Advanced homework: glossary data (replaces the FAQ data cell) ----
# Upload to /content/ first: glossary_sft.jsonl (output of scripts/make_sft_dataset.py) and data/glossary.json
import json, re
from pathlib import Path

GLOSSARY_PATH, SFT_PATH = "/content/glossary.json", "/content/glossary_sft.jsonl"
terms = json.loads(Path(GLOSSARY_PATH).read_text(encoding="utf-8"))["terms"]
DEF   = {t["term"]: t["definition"] for t in terms}
ALIAS = {t["term"]: t.get("aliases", []) for t in terms}
KNOWN = {n.lower() for t in terms for n in [t["term"], *t.get("aliases", [])]}

UNGROUNDED_ANSWER = "This glossary does not contain that information. Confirmation from the course organizers is needed."
FAQ_TEXT = ""   # closed-book: no reference text is given to the model (kept so old cells don't break)

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

# ---- 1) Grounded training data = output of make_sft_dataset.py (plain definitions) wrapped in the JSON format ----
TEMPLATES = ["What is {term}?", "Briefly explain {term}."]
if Path(SFT_PATH).exists():
    sft_rows = [json.loads(l) for l in Path(SFT_PATH).read_text(encoding="utf-8").splitlines() if l.strip()]
    pairs = [(r["messages"][0]["content"], r["messages"][1]["content"]) for r in sft_rows]
else:   # same thing the script produces
    pairs = [(tp.format(term=t["term"]), t["definition"]) for t in terms for tp in TEMPLATES]
print("SFT pairs from script:", len(pairs))

# validation: hold out the 'Briefly explain' variant for 6 terms (only for loss monitoring)
VALID_TERMS = ["Branch", "Prompt", "API key", "Guardrail", "Structured output", "Pull Request (PR)"]
valid_qs = {f"Briefly explain {t}." for t in VALID_TERMS}
train, valid, test, gold, KIND = [], [], [], {}, {}
for q, a in pairs:
    (valid if q in valid_qs else train).append(make_example(q, a, False))

# ---- 2) Ungrounded examples: terms that are NOT in the glossary (train / valid / test terms are all different) ----
TRAIN_OOD = ["Mixture of Experts", "Prompt injection", "Vector database", "Knowledge distillation", "Gradient checkpointing", "Batch size",
             "Dropout", "Attention mechanism", "Tokenizer", "Beam search", "Reinforcement learning", "Transformer"]
VALID_OOD = ["Layer normalization", "Cosine annealing"]
TEST_OOD  = ["KV cache", "DoRA", "Retrieval-augmented generation", "RLHF", "Chain of thought", "Prompt caching", "Few-shot prompting", "Weight decay"]
for t in TRAIN_OOD + VALID_OOD + TEST_OOD:
    assert t.lower() not in KNOWN, f"{t} is in the glossary - pick another"
for t in TRAIN_OOD:
    for tp in TEMPLATES: train.append(make_example(tp.format(term=t), UNGROUNDED_ANSWER, True))
for t in VALID_OOD: valid.append(make_example(f"What is {t}?", UNGROUNDED_ANSWER, True))

# ---- 3) Test set: 4 kinds. must = a phrase copied from the definition (hand-checked below) ----
MUST = {"LoRA": "low-rank product", "Token": "smallest unit", "Epoch": "One pass", "Needs review": "human review",
        "Rate limit": "HTTP 429", "Human-in-the-loop": "requires_approval=True", "Overfitting": "fits training data excessively",
        "Learning rate": "how far parameters move", "Harness": "four built-in tools", "Hallucination": "plausible content",
        "Quantization": "fewer bits", "Rubric": "pass/fail conditions"}
for t, m in MUST.items(): assert m in DEF[t], (t, m)           # required phrase really is in the definition
REPHRASE = ["I keep hearing about {term}. What does it mean?", "Define {term} as it is used in this course."]   # NOT used in training
ALIAS_TESTS = {"LoRA": "Low-Rank Adaptation", "Hallucination": "fabrication", "Rate limit": "429 error", "Human-in-the-loop": "HITL",
               "Rubric": "grading criteria", "Learning rate": "lr"}
DESCRIPTIVE = [  # the term is never named
    ("Which method freezes pretrained weights and trains only small low-rank matrices?", "LoRA"),
    ("What do you call one full pass through the entire training dataset?", "Epoch"),
    ("What is the term for an LLM producing plausible but factually wrong content?", "Hallucination"),
    ("What is it called when a model fits its training data too well and does worse on unseen inputs?", "Overfitting")]

def add_test(q, kind, term=None):
    if term is None:
        test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["Confirmation"]}
    else:
        test.append(make_example(q, DEF[term], False));        gold[q] = {"needs_review": False, "must_include": [MUST[term]]}
    KIND[q] = kind

for i, t in enumerate(MUST):
    add_test(f"What is {t}?", "direct", t)                           # same wording as training
    add_test(REPHRASE[i % 2].format(term=t), "rephrase", t)          # new wording
for t, al in ALIAS_TESTS.items():
    assert al in ALIAS[t], (t, al); add_test(f"What is {al}?", "alias", t)
for q, t in DESCRIPTIVE: add_test(q, "descriptive", t)
for t in TEST_OOD: add_test(f"What is {t}?", "ungrounded")

# Only the "direct" kind may reuse a training question on purpose (it measures recall of seen wording); everything else must be unseen
tq = {r["question"] for r in train}; vq = {r["question"] for r in valid}; teq = [r["question"] for r in test]
assert len(set(teq)) == len(teq) and not (vq & set(teq)) and not (vq & tq), "duplicate questions"
assert all(KIND[q] == "direct" for q in teq if q in tq), "a non-direct test question appears in training"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}", {k: sum(v == k for v in KIND.values()) for k in dict.fromkeys(KIND.values())})
print("Example:", train[0]["question"], "->", train[0]["target"][:120], "...")
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")


SFT pairs from script: 130
train=148 valid=8 test=42 gold=42 {'direct': 12, 'rephrase': 12, 'alias': 6, 'descriptive': 4, 'ungrounded': 8}
Example: What is AI engineering? -> {"answer": "The technical field of designing, building, evaluating, and operating useful applications and systems using  ...


## 3. Load the Model and Evaluate Before Training

**What to Do**: Load the base model without quantization (32-bit), have it answer the 21 test questions, and score the results.

The prompt (`SYSTEM`) says "Always return `answer` and `needs_review` in one JSON line". The **format has already been instructed**. Before training, check whether the model follows this instruction.

**Scoring** (the same functions are used in section 5)

- **Format**: The output parses as JSON, has exactly two keys (`answer` and `needs_review`), and uses string and boolean types respectively.
- **Content**: `needs_review` matches the expected answer, and `answer` contains the required keywords (for example, "September 30" and "23:59").

Content scoring requires a matching `needs_review` value. Outputs with a missing key therefore receive `Fail` for content even if the wording is correct. Check the wording yourself in the table. The format-failure codes mean: `json_error` = JSON parsing failed; `not_object` = the JSON value is not an object; `extra_keys` = unexpected keys are present; `missing_key` = a required key is absent; `wrong_type` = an answer or review flag has the wrong type.

`Qwen/Qwen3.5-0.8B` also supports images, but loading it with `AutoModelForCausalLM` uses only the text component.

**What to Look For**

- First cell: a message such as `Weight memory (GB): 3.01`. Downloading the model (about 1.75GB) takes about 1 minute.
- Second cell: a one-line summary (`format_rate`, `content_rate`, etc.) and a table of outputs for 21 questions. Evaluation takes about 1 minute. In the original Japanese run, many before-training outputs were missing `needs_review`; check which keys are missing in your run.

**Harmless Warnings**

- `You are sending unauthenticated requests to the HF Hub`: this means the model is being downloaded without signing in to Hugging Face. Public models can be downloaded without signing in. Signing in is needed in the following cases, regardless of model size:
  - Using a model that requires agreement to terms (a gated model, such as Llama or Gemma) or a private repository.
  - Uploading your own models or data to the Hub.
  - Sending many requests in a short time and hitting a limit. Without signing in, the limit is 3,000 file retrievals per IP address per 5 minutes (published figure as of September 2025; [Hub rate limits](https://huggingface.co/docs/hub/rate-limits)). Signing in raises it.

  This notebook's model is public and requires no agreement. Loading it makes at most a few dozen requests, well below the limit. If `429 Too Many Requests` does occur, the library reads the waiting time and retries automatically. If signing in becomes necessary, register a Hugging Face token named `HF_TOKEN` using the key icon (Secrets) on Colab's left sidebar and allow this notebook to access it; it will be used automatically.
- `... is falling back to its reference PyTorch implementation ...`: a fast implementation for some of this model's layers (linear attention) is not installed, so a slower but correct implementation is used. About 4 lines appear.

**Read Before Continuing**: Check the table for "which keys are missing" and "whether the wording agrees with the FAQ". In section 5, compare this table alongside outputs after training. Original outputs are also saved in `/content/qlora_lab/eval_before.json` (open it from the file list on the left).

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# No quantization. Keep weights in 32-bit (specify dtype because omitting it loads the distributed precision)
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| Weight memory (GB):", BASE_MEMORY_GB)

# The prompt instructs the format. Before training, check whether the model follows it
SYSTEM = "You answer questions about the AIE2026 course glossary from memory. Always answer in one JSON line. Format: {\"answer\": string, \"needs_review\": boolean}. If the term is not in the course glossary, write 'Confirmation from the course organizers is needed' in answer and set needs_review to true."

def build_messages(example):
    # Closed book: only the question is given (no reference text), so knowledge has to come from the weights
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"Question: {example['question']}"}]

def generate(model, example):
    # enable_thinking=False: leave the thinking block empty and generate only the answer (explicit for consistent conditions across models)
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | Weight memory (GB): 3.01


In [7]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # Display original outputs without truncation

def check_format(text: str):
    '''Check JSON parsing, required keys, and types. Also return the reason for failure'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''Content passes if needs_review matches the expected answer and answer contains required keywords'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "Pass" if ok else "Fail"

def evaluate(model, label: str):
    '''Generate and score the 21 test questions, returning a summary and per-question results. Save original outputs in eval_<label>.json'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # If the format is broken but a JSON-like portion can be parsed, score content separately
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# Per-question outputs. Compare with expected needs_review and check for missing keys and agreement with FAQ wording
display(pd.DataFrame([{"Question": r["question"], "Expected needs_review": r["gold_needs_review"], "Before-training output": r["raw"],
                       "Format": mark(r["format_ok"]), "Format failure reason": r["format_reason"], "Content": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 42, "format_rate": 1.0, "content_rate": 0.0, "both_rate": 0.0, "seconds": 172.1}


,Question,Expected needs_review,Before-training output,Format,Format failure reason,Content
0,What is LoRA?,False,"{""answer"": ""LoRA stands for Low-Rank Adaptation, a technique used in Generative AI to fine-tune large pre-trained language models (like BERT) with minimal computational cost and limited memory usage. It involves creating a small, trainable subset of weights from the original model and fine-tuning them on a specific dataset, allowing the model to learn new patterns while retaining the ability to generate new content."", ""needs_review"": false}",Pass,,Fail
1,I keep hearing about LoRA. What does it mean?,False,"{""answer"": ""LoRA stands for \""Low-Rank Adaptation\"". It is a technique in the realm of Large Language Models (LLMs) that allows a model to be fine-tuned on a small dataset of data, while retaining its original knowledge and capabilities. This is achieved by using a small, pre-trained model as a \""base\"" and then adding a small, trainable layer to the end of the model's architecture. This allows the model to learn new patterns and knowledge from a small amount of data without losing its ability to understand and generate the original content."", ""needs_review"": false}",Pass,,Fail
2,What is Token?,False,"{""answer"": ""A token is a unit of currency used in the AIE2026 course glossary."", ""needs_review"": true}",Pass,,Fail
3,Define Token as it is used in this course.,False,"{""answer"": ""A Token is a unit of measurement used in the AIE2026 course to quantify the amount of energy or power delivered to a specific component or system. It is typically expressed in units such as Watts (W), Joules (J), or kilowatt-hours (kWh), depending on the context of the specific course module."", ""needs_review"": false}",Pass,,Fail
4,What is Epoch?,False,"{""answer"": ""Epoch is a time period in astronomy and physics, typically defined as the time it takes for the Earth to complete one orbit around the Sun."", ""needs_review"": false}",Pass,,Fail
5,I keep hearing about Epoch. What does it mean?,False,"{""answer"": ""Epoch is a term used in astronomy to describe a specific time period, typically a year, during which a celestial body (like a planet or moon) is in a specific state of orbit or rotation."", ""needs_review"": false}",Pass,,Fail
6,What is Needs review?,False,"{""answer"": ""Needs review is a process where a project team reviews the requirements and constraints of a project to ensure they are met and that the project is feasible and achievable."", ""needs_review"": true}",Pass,,Fail
7,Define Needs review as it is used in this course.,False,"{""answer"": ""Needs review is a process where the course organizers review the course syllabus, curriculum, and learning outcomes to ensure they align with the course objectives and meet the specific learning outcomes of the AIE2026 course."", ""needs_review"": true}",Pass,,Fail
8,What is Rate limit?,False,"{""answer"": ""Rate limit is a mechanism used to control the number of requests a server can process within a specific time period, typically per IP address or per IP address per second. It prevents abuse by ensuring that a server does not overload itself with too many requests, which could lead to service degradation or even failure."", ""needs_review"": false}",Pass,,Fail
9,I keep hearing about Rate limit. What does it mean?,False,"{""answer"": ""Rate limit is a security feature that restricts the number of requests a server can process within a specific time period or per IP address. It helps prevent abuse, such as DoS attacks or spam, by ensuring that legitimate users are not overwhelmed by excessive requests."", ""needs_review"": false}",Pass,,Fail


## 4. LoRA Training (Original Japanese Run: 4–5 Minutes on a T4)

### LoRA versus QLoRA

- **LoRA**: Freeze the base-model weights and train only small matrices (adapters) added to some layers.
- **QLoRA**: Keep base-model weights quantized to 4-bit and train only adapters in the same way. The adapters themselves are not quantized.

QLoRA reduces the memory used to hold base-model weights. It uses about a quarter of the memory of 16-bit, so models too large to fit on a GPU otherwise can be trained (for 7 billion parameters, about 14GB at 16-bit versus about 3.5GB at 4-bit).

| Aspect | LoRA (no quantization) | QLoRA (4-bit) |
| --- | --- | --- |
| Weight memory | Large | Small (about 1/4 of 16-bit) |
| Base-model outputs | Unchanged | May change due to quantization error; the size of the effect depends on the model |
| Training speed | Baseline | Not faster. Often slower because 4-bit weights must be converted back for each computation |
| Dependencies | peft | peft plus bitsandbytes (requires a CUDA GPU) |

Notes for implementing QLoRA:

- Pass `BitsAndBytesConfig(load_in_4bit=True, ...)` when loading, then run `prepare_model_for_kbit_training(model)` before adding LoRA.
- Storage precision (4-bit) and compute precision (`bnb_4bit_compute_dtype`, such as 16-bit) are separate settings.
- Compare before and after training using the same quantization conditions for the base model. Measuring without quantization before training and with 4-bit afterward mixes quantization and training effects.
- Adapters are created for the base model used in training. Record the base-model ID, revision, and quantization settings together when distributing or reusing them.
- 4-bit training is discouraged for some models. One [training-library guide](https://unsloth.ai/docs/models/qwen3.5/fine-tune) discourages 4-bit training of Qwen3.5 because its quantization error is larger than usual.

This exercise's model fits on a T4 without quantization, so we train with LoRA. Consider LoRA first if the model fits; if it does not, consider QLoRA or a smaller model. You can examine memory and output differences when loading in 4-bit in section 8 (Advanced).

### Run Training

**What to Do**: Convert the 14 train examples to chat format (system / user / assistant), attach LoRA, and train for 20 epochs (40 steps). Save the trained adapter in `adapter/`.

- `target_modules` specifies layers where LoRA is inserted. We target attention projections (such as `q_proj`) and MLP projections (such as `gate_proj`). This model uses linear attention in 18 of its 24 layers; only the remaining 6 have `q_proj` and similar projections. All layers have MLPs.
- Batch 4 × gradient accumulation 2 = 8 examples per step. With 14 train examples, each epoch has 2 steps (8 and 6 examples), giving 40 steps over 20 epochs. The model sees each training example exactly 20 times. Learning-rate warmup and cosine decay apply over these 40 steps.
- The trained model is stored in `tuned`. Since LoRA is inserted into the base-model layers, generation through the original variable `model` also includes adapters. Temporarily disable adapters as in section 6 when you need before-training outputs.

**What to Look For**

- `trainable params: ... || all params: ... || trainable%: ...`: a feature of LoRA is that only a few percent of all parameters are trained.
- The `Training Loss` table (every 10 steps): in the original Japanese run, loss started around 1.5 and dropped to 0.01–0.02 by step 40. The translated data may give different values. If training ends without a decrease, check whether you changed the settings.
- `Training time: ... seconds | Steps: 40 | Epochs: 20.0`: the original Japanese run took 240–290 seconds on a T4; timing may differ with the translated data. Steps and epochs confirm that 14 training examples × 20 passes were run as calculated above.

**Harmless Warning**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` means end-token settings were aligned with the tokenizer.

**Troubleshooting**: If training stops midway or cannot run, read 4b below.

In [8]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # Training data pairs inputs (system + user) with desired outputs (assistant). Use the same build_messages as for evaluation
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 uses linear attention in 18 of 24 layers; only the remaining 6 have q/k/v/o_proj. All layers have gate/up/down_proj (MLP)
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # Normally leave unchanged. True enables 16-bit mixed-precision computation (CUDA only; weights stay 32-bit). Faster, but loss may become nan
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # Fraction of parameters trained with LoRA
t0 = time.time()
trainer.train()  # See 4b to resume if interrupted
print(f"Training time: {time.time()-t0:.0f} seconds | Steps: {trainer.state.global_step} | Epochs: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # Training mode enables dropout and makes evaluation outputs vary (assignment suppresses displaying the whole model)

Tokenizing train dataset:   0%|          | 0/148 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/148 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/148 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/148 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.298998
20,0.984619
30,0.774662
40,0.518942
50,0.338215
60,0.213955
70,0.113092
80,0.093743
90,0.066954
100,0.052242


Training time: 613 seconds | Steps: 152 | Epochs: 8.0


### 4b. If Training Stops or Cannot Run

**Stopped midway (runtime disconnected, etc.)**: Checkpoints are saved every 10 steps in `/content/qlora_lab/ckpt/`. If the files remain, rerun sections 0–3, then change `trainer.train()` in the training cell above to `trainer.train(resume_from_checkpoint=True)` and run it. Training resumes from the last checkpoint up to 40 steps (20 epochs). If the runtime and its files are gone, restart training or use the alternative below.

**Could not train (no GPU available or not enough time)**: Compare and discuss the instructor's results (outputs and scores after training) from the original Japanese notebook run on a T4. Cells from section 5 onward require the trained model `tuned`, so read the explanations alongside the instructor's results. When submitting, set `executed_training_myself` in section 7 to `False` and state clearly how far you ran the notebook yourself.

## 5. Evaluate After Training (Same test and Generation Settings as Before)

**What to Do**: Evaluate the trained model `tuned` on exactly the same 21 questions and generation settings, and compare before and after. The only change is whether adapters are present.

**What to Look For**

- Comparison table: how did `format_rate` (format compliance rate) and `content_rate` (content accuracy) change from before training?
- `Regressions`: questions that passed both format and content before training but failed afterward. Ideally the list is empty; inspect any regressions in your own run.
- Per-question table: read the remaining failures and identify their types. The original Japanese run included these types:
  - Applying a different FAQ item and making an assertion for a question without evidence (expected `needs_review` is `True`).
  - Evading a question with evidence (expected value is `False`) by requesting confirmation and saying "It is not listed".

In the original Japanese run, after-training answers tended to reproduce the training text (the original FAQ wording) almost verbatim. Check whether your outputs show the same pattern. Format and wording become consistent, but the model has not necessarily learned to judge "whether the FAQ contains evidence".

GPU training is not perfectly reproducible even with a fixed seed. Format compliance and content accuracy may be nearly the same while the remaining questions differ between runs. It is normal for your "remaining questions" to differ from your classmate's; compare failure types.

In [9]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# Questions that passed both format and content before training but failed afterward (regressions)
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("Regressions:", regressed)

# Original per-question outputs. Visually check format (JSON, required keys, types) and content (matching needs_review and required keywords) judgments
detail = pd.DataFrame([{"Question": b["question"], "Expected needs_review": b["gold_needs_review"],
                        "Before-training output": b["raw"], "Before format/content": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "After-training output": a["raw"], "After format/content": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 42, "format_rate": 1.0, "content_rate": 0.6666666666666666, "both_rate": 0.6666666666666666, "seconds": 146.0}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,42,1.0,0.000000,0.000000,172.1
after,42,1.0,0.666667,0.666667,146.0


Regressions: []


,Question,Expected needs_review,Before-training output,Before format/content,After-training output,After format/content
0,What is LoRA?,False,"{""answer"": ""LoRA stands for Low-Rank Adaptation, a technique used in Generative AI to fine-tune large pre-trained language models (like BERT) with minimal computational cost and limited memory usage. It involves creating a small, trainable subset of weights from the original model and fine-tuning them on a specific dataset, allowing the model to learn new patterns while retaining the ability to generate new content."", ""needs_review"": false}",Pass/Fail,"{""answer"": ""A PEFT method that freezes pretrained weights, represents weight updates as a low-rank product of two small matrices, and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA."", ""needs_review"": false}",Pass/Pass
1,I keep hearing about LoRA. What does it mean?,False,"{""answer"": ""LoRA stands for \""Low-Rank Adaptation\"". It is a technique in the realm of Large Language Models (LLMs) that allows a model to be fine-tuned on a small dataset of data, while retaining its original knowledge and capabilities. This is achieved by using a small, pre-trained model as a \""base\"" and then adding a small, trainable layer to the end of the model's architecture. This allows the model to learn new patterns and knowledge from a small amount of data without losing its ability to understand and generate the original content."", ""needs_review"": false}",Pass/Fail,"{""answer"": ""A PEFT method that freezes pretrained weights, represents weight updates as a low-rank product of two small matrices, and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA."", ""needs_review"": false}",Pass/Pass
2,What is Token?,False,"{""answer"": ""A token is a unit of currency used in the AIE2026 course glossary."", ""needs_review"": true}",Pass/Fail,"{""answer"": ""The smallest unit an LLM uses to process text. Text is split into fragments of words or characters. API charges, rate limits, and context window sizes are often counted in tokens. Tokenization varies by language; for example, Japanese tends to require more tokens per character than English."", ""needs_review"": false}",Pass/Pass
3,Define Token as it is used in this course.,False,"{""answer"": ""A Token is a unit of measurement used in the AIE2026 course to quantify the amount of energy or power delivered to a specific component or system. It is typically expressed in units such as Watts (W), Joules (J), or kilowatt-hours (kWh), depending on the context of the specific course module."", ""needs_review"": false}",Pass/Fail,"{""answer"": ""The smallest unit an LLM uses to process text. Text is split into fragments of words or characters. API charges, rate limits, and context window sizes are often counted in tokens. Tokenization varies by language; for example, Japanese tends to require more tokens per character than English."", ""needs_review"": false}",Pass/Pass
4,What is Epoch?,False,"{""answer"": ""Epoch is a time period in astronomy and physics, typically defined as the time it takes for the Earth to complete one orbit around the Sun."", ""needs_review"": false}",Pass/Fail,"{""answer"": ""One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting."", ""needs_review"": false}",Pass/Pass
5,I keep hearing about Epoch. What does it mean?,False,"{""answer"": ""Epoch is a term used in astronomy to describe a specific time period, typically a year, during which a celestial body (like a planet or moon) is in a specific state of orbit or rotation."", ""needs_review"": false}",Pass/Fail,"{""answer"": ""One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting."", ""needs_review"": false}",Pass/Pass
6,What is Needs review?,False,"{""answer"":

## 5b. Advanced: per-kind comparison and a glossary-search baseline

Search = look the question up in the glossary (TF-IDF over term + aliases + definition, with an exact-title boost); answer with the stored definition, or `needs_review: true` when the best match scores below a threshold chosen on the **validation** questions. No training. Scored with the same `check_format` / `check_content`.


In [10]:
# ---- Glossary SEARCH baseline: no training, no GPU. Same test set, same scoring ----
import math, collections
STOP = set("what is are the a an of to in on for and or me i you it its this that as how do does did can by with from be use used which who "
           "call called term name keep hearing about mean define explain briefly course one full".split())
def toks(s): return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP and len(w) > 1]
docs = {t["term"]: toks(" ".join([t["term"], *t.get("aliases", []), t["definition"]])) for t in terms}
N = len(docs); df = collections.Counter(w for d in docs.values() for w in set(d))
def idf(w): return math.log((N + 1) / (df.get(w, 0) + 0.5))
def title_hit(q, t):   # exact term / alias in the question (word boundaries)
    return any(re.search(r"(?<![a-z0-9])" + re.escape(n.lower()) + r"(?![a-z0-9])", q.lower()) for n in [t["term"], *t.get("aliases", [])])
def search_scores(q):
    qt = set(toks(q)); out = []
    for t in terms:
        d = collections.Counter(docs[t["term"]])
        s = sum(idf(w) * (1 + math.log(d[w])) for w in qt if w in d) + (6.0 if title_hit(q, t) else 0.0)
        out.append((s, t["term"]))
    return sorted(out, reverse=True)
# Threshold chosen on the VALIDATION questions only (never on test)
g_scores = [search_scores(e["question"])[0][0] for e in valid if not e["needs_review"]]
o_scores = [search_scores(e["question"])[0][0] for e in valid if e["needs_review"]]
SEARCH_THRESH = (max(o_scores) + min(g_scores)) / 2 if max(o_scores) < min(g_scores) else max(o_scores) + 0.01
print(f"validation scores: ungrounded max={max(o_scores):.2f}, grounded min={min(g_scores):.2f} -> threshold {SEARCH_THRESH:.2f}")
def search_answer(q):
    s, best = search_scores(q)[0]
    if s >= SEARCH_THRESH: return json.dumps({"answer": DEF[best], "needs_review": False}, ensure_ascii=False)
    return json.dumps({"answer": UNGROUNDED_ANSWER, "needs_review": True}, ensure_ascii=False)

def score_rows(raw_by_q):
    rows = []
    for ex in test:
        raw = raw_by_q[ex["question"]]; ok_fmt, reason, obj = check_format(raw)
        ok_c = check_content(obj, gold[ex["question"]])
        rows.append({"question": ex["question"], "kind": KIND[ex["question"]], "raw": raw, "format_ok": ok_fmt, "content_ok": ok_c,
                     "both_ok": ok_fmt and ok_c, "gold_needs_review": gold[ex["question"]]["needs_review"],
                     "asserted": bool(ok_fmt and obj["needs_review"] is False and gold[ex["question"]]["needs_review"])})
    return rows
search_rows = score_rows({ex["question"]: search_answer(ex["question"]) for ex in test})
(WORK / "eval_search.json").write_text(json.dumps(search_rows, ensure_ascii=False, indent=2), encoding="utf-8")
print("search baseline: content_rate =", round(sum(r["content_ok"] for r in search_rows) / len(search_rows), 3))


validation scores: ungrounded max=0.00, grounded min=10.55 -> threshold 5.27
search baseline: content_rate = 0.976


In [11]:
# ---- Per-kind comparison: base model vs LoRA-tuned vs glossary search ----
def by_kind(rows, with_kind=True):
    out = {}
    for k in list(dict.fromkeys(KIND.values())) + ["ALL"]:
        rs = [r for r in rows if k == "ALL" or KIND[r["question"]] == k]
        out[k] = {"n": len(rs), "format": round(sum(r["format_ok"] for r in rs) / len(rs), 2), "content": round(sum(r["content_ok"] for r in rs) / len(rs), 2)}
    return out
asserted_ood = lambda rows: sum(1 for r in rows if KIND[r["question"]] == "ungrounded" and r["format_ok"] and json.loads(r["raw"])["needs_review"] is False)
tbl = {}
for name, rows in [("before (base)", before_rows), ("after (LoRA)", after_rows), ("glossary search", search_rows)]:
    d = by_kind(rows); tbl[name] = {**{f"{k} content": v["content"] for k, v in d.items()}, "format ALL": d["ALL"]["format"], "ungrounded asserted": asserted_ood(rows)}
display(pd.DataFrame(tbl).T)
# Failure inspection: what did the tuned model get wrong?
fails = [r for r in after_rows if not r["both_ok"]]
display(pd.DataFrame([{"kind": KIND[r["question"]], "question": r["question"], "expected needs_review": r["gold_needs_review"], "after output": r["raw"]} for r in fails]))


,direct content,rephrase content,alias content,descriptive content,ungrounded content,ALL content,format ALL,ungrounded asserted
before (base),0.0,0.0,0.00,0.0,0.00,0.00,1.0,6.0
after (LoRA),1.0,1.0,0.17,0.0,0.38,0.67,1.0,5.0
glossary search,1.0,1.0,1.00,1.0,0.88,0.98,1.0,1.0


,kind,question,expected needs_review,after output
0,alias,What is Low-Rank Adaptation?,False,"{""answer"": ""This glossary does not contain that information. Confirmation from the course organizers is needed."", ""needs_review"": true}"
1,alias,What is fabrication?,False,"{""answer"": ""The art of creating content that appears authentic through the manipulation of external information, such as updating data, changing author names, or adding fluff. It is a major reason for grading inflation."", ""needs_review"": false}"
2,alias,What is 429 error?,False,"{""answer"": ""An error returned when the model calls tools or reads files with permissions not granted by the runner. It occurs when the model tries to write to a location the runner has not allowed, such as /app/output or a file the runner does not own. The easiest fix is name the output file something the runner can own, for example, output.txt."", ""needs_review"": false}"
3,alias,What is HITL?,False,"{""answer"": ""A ready-made LLM application that runs in the terminal. It has the same API as a free model, but includes tools, a chat interface, and a file editor. Test by running it with: pip install hitl and then python hitl.py. It is designed for doing repetitive work such as coding, writing, and documentation."", ""needs_review"": false}"
4,alias,What is grading criteria?,False,"{""answer"": ""A document listing the expected outputs, format, and conditions for passing for each exercise. Located at exercises/NN/criteria.md. Passing this document indicates that the code implements the exercise's goal; passing the grade indicates the implementation meets the requirements."", ""needs_review"": false}"
5,descriptive,Which method freezes pretrained weights and trains only small low-rank matrices?,False,"{""answer"": ""Contraction (or LoRA) is a method that freezes pretrained weights and trains only small low-rank matrices. It is used for when the model has not grown sufficiently since freezing, or when training for a very short time. LoRA has many advantages, including being able to train multiple models in parallel, using only a small amount of GPU memory, and avoiding OOM errors."", ""needs_review"": false}"
6,descriptive,What do you call one full pass through the entire training dataset?,False,"{""answer"": ""Iterative training. It is more suitable for large datasets with many questions and answers. Two agents—one for questioning and one for answering—can handle this."", ""needs_review"": false}"
7,descriptive,What is the term for an LLM producing plausible but factually wrong content?,False,"{""answer"": ""LoRA align (LORA-Align). It is a method of fine-tuning LoRA adapters to make the model align more closely to instructions. Even when used correctly, it can still produce plausible but factually wrong content."", ""needs_review"": false}"
8,descriptive,What is it called when a model fits its training data too well and does worse on unseen inputs?,False,"{""answer"": ""Overfitting. Detect it by measuring performance on data not used for training, such as questions about the course glossary."", ""needs_review"": false}"
9,ungrounded,What is KV cache?,True,"{""answer"": ""A mechanism for storing previous model outputs (and intermediate results) in memory. This lets the model request additional tokens closer to its generation limit, rather than sending them to the LLM. Both LLM and MCP clients can use it. In Pydantic AI, it is provided by the Agent tool with name \""memory\""."", ""needs_review"": false}"


## 6. Knowledge Questions (Learning a Format Does Not Guarantee Knowledge)

**What to Do**: Separately from the format experiment, ask 3 factual questions absent from both the FAQ and training data (plus one FAQ-covered question as a control). The desired response is "Confirmation is needed (`needs_review: true`)".

Classify responses into three categories.

| Classification | Meaning |
| --- | --- |
| `format_error` | Does not follow the JSON format (common before training in the original Japanese run) |
| `needs_review` | Returns "It is not listed. Confirmation is needed" (the desired response) |
| `asserted` | Makes an assertion with `needs_review: false`. For a question absent from the FAQ, this is an unsupported assertion (fabrication) |

**What to Look For**: In the original Japanese run, format errors were common before training, and both "confirmation needed" and "assertion" responses appeared after training. Check the response types in your own run. The table also shows original outputs. For assertions, read what the model seems to rely on (whether it reuses a different FAQ item). Report results for this dataset rather than deciding in advance that "fine-tuning can never correct knowledge".

Before-training outputs are obtained by temporarily disabling adapters (`tuned.disable_adapter()`).

In [12]:
KNOWLEDGE_QUESTIONS = [
    "What is the instructor's name?",                         # Absent from glossary and training
    "What is the topic of the next lecture?",                 # Absent
    "When is the final assignment deadline for this course?", # Absent
    "What is LoRA?",                                          # In glossary (control, seen in training)
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

grounded_questions = {ex["question"] for ex in train + valid + test if not ex["needs_review"]}
knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA is inserted into base-model layers, so generation with `model` also uses adapters after training. Temporarily disable adapters for before-training outputs
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"Question": q, "In FAQ": q in grounded_questions, "Before-training classification": classify(before_raw), "Before-training output": before_raw,
                 "After-training classification": classify(after_raw), "After-training output": after_raw})
display(pd.DataFrame(knowledge_rows))

,Question,In FAQ,Before-training classification,Before-training output,After-training classification,After-training output
0,What is the instructor's name?,False,asserted,"{""answer"": ""Dr. Sarah Jenkins"", ""needs_review"": false}",asserted,"{""answer"": ""Alexio Maurensi"", ""needs_review"": false}"
1,What is the topic of the next lecture?,False,asserted,"{""answer"": ""The next lecture will cover the latest advancements in AI ethics and governance."", ""needs_review"": false}",asserted,"{""answer"": ""Evaluation and improvement of evaluation methods and datasets. Discuss how to choose a problem, select a grading rubric, and interpret graders' answers. Also cover AI-generated content, such as writing and demonstrations, and how to detect them."", ""needs_review"": false}"
2,When is the final assignment deadline for this course?,False,asserted,"{""answer"": ""The final assignment deadline is 10:00 PM on Friday, October 25, 2026."", ""needs_review"": false}",asserted,"{""answer"": ""31 December 2026. Late submissions will not be graded."", ""needs_review"": false}"
3,What is LoRA?,True,asserted,"{""answer"": ""LoRA stands for Low-Rank Adaptation, a technique used in Generative AI to fine-tune large pre-trained language models (like BERT) with minimal computational cost and limited memory usage. It involves creating a small, trainable subset of weights from the original model and fine-tuning them on a specific dataset, allowing the model to learn new patterns while retaining the ability to generate new content."", ""needs_review"": false}",asserted,"{""answer"": ""A PEFT method that freezes pretrained weights, represents weight updates as a low-rank product of two small matrices, and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA."", ""needs_review"": false}"


## 7. Comparison Table and Submission

**What to Do**: Combine settings, library versions, before/after summaries, and regressed questions into one `result.json`.

**What to Submit**

- `result.json`, `eval_before.json`, and `eval_after.json` (in `/content/qlora_lab`; right-click in the file list on the left → **Download**. With `USE_DRIVE = True`, they are also copied to Drive)
- The comparison table from section 5 and an explanation of remaining failures (failure types)
- Results from section 6 (responses that changed to confirmation requests and those that remained assertions)
- How far you ran training yourself. If you used the instructor's results, set `executed_training_myself` to `False`.

In [13]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (no quantization), trained on glossary SFT data (Advanced homework)",
    "per_kind": tbl, "train_size": len(train), "test_size": len(test),
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # Set False if using the instructor's results
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Saved to Drive:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 8,
    "warmup_steps": 10,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 160,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.14.1+cu130"
  },
  "method": "LoRA (no quantization), trained on glossary SFT data (Advanced homework)",
  "per_kind": {
    "before (base)": {
      "direct content": 0.0,
      "rephrase content": 0.0,
      "alias content": 0.0,
      "descriptive content": 0.0,
      "ungrounded content": 0.0,
      "ALL content": 0.0,
      "format ALL": 1.0,
      "ungrounded asserted": 6
    },
    "after (LoRA)": {
      "direct content": 1.0,
      "rephrase content": 1.0,
      "alias content": 0.17,
      "descriptive content": 0.0,
      "ungrounded cont

## 8. Advanced: Load with 4-bit Quantization and Compare Memory and Outputs (Optional)

**What to Do**: Load the same model in 4-bit (NF4) and observe changes in weight memory and before-training outputs. QLoRA adds LoRA to this 4-bit base model and trains it. No training is done here.

- bitsandbytes requires a CUDA GPU, so this runs only in a GPU runtime. Installation takes about 1 minute.
- 4-bit loading places the model on the GPU with `device_map="auto"` (matching bitsandbytes' loading method).
- If you also try training the 4-bit model, run `prepare_model_for_kbit_training(model_4bit)` first, follow section 4's training procedure, and evaluate both before and after using a 4-bit base model.

**What to Look For**

- Weight memory: in the original Japanese run, 4-bit used less than 1GB, versus about 3GB at 32-bit. Compare the displayed values in your run.
- Comparison and per-question tables: since no training occurred, differences from the 32-bit before-training results (`before`) reflect quantization. In the original Japanese run, some questions improved, while other outputs had `needs_review` as the string `"false"` and failed type checks, or asserted "yes" or "none" for questions without evidence. Check the kinds and direction of changes in your run.

In [14]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # Storage precision: 4-bit (NF4)
                         bnb_4bit_compute_dtype=torch.float16,             # Compute precision: 16-bit (T4 does not support bf16)
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("Weight memory (GB) | No quantization (32-bit):", BASE_MEMORY_GB, "| 4-bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# No training occurred, so differences from the unquantized before-training results (before) reflect quantization
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"Question": r["question"], "Expected needs_review": r["gold_needs_review"], "4-bit before-training output": r["raw"],
                       "Format/content": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "Format failure reason": r["format_reason"]} for r in q_rows]))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Weight memory (GB) | No quantization (32-bit): 3.01 | 4-bit: 0.76
{"label": "before_4bit", "n": 42, "format_rate": 1.0, "content_rate": 0.0, "both_rate": 0.0, "seconds": 154.3}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,42,1.0,0.0,0.0,172.1
before_4bit,42,1.0,0.0,0.0,154.3


,Question,Expected needs_review,4-bit before-training output,Format/content,Format failure reason
0,What is LoRA?,False,"{""answer"": ""LoRA stands for Low-Rank Adaptation, a technique used in deep learning to improve model performance by adding small, trainable modifications to a pre-trained model."", ""needs_review"": false}",Pass/Fail,
1,I keep hearing about LoRA. What does it mean?,False,"{""answer"": ""LoRA stands for \""Low-Resource Augmented Representation\"". It is a technique in the realm of Large Language Models (LLMs) that allows a model to be trained on a smaller subset of data compared to the full dataset, while still maintaining high performance on the larger dataset."", ""needs_review"": false}",Pass/Fail,
2,What is Token?,False,"{""answer"": ""Token is a data structure in the AIE2026 course glossary that represents a specific type of data object used in the system."", ""needs_review"": true}",Pass/Fail,
3,Define Token as it is used in this course.,False,"{""answer"": ""A Token is a small, reusable piece of data that is used to identify and authenticate a user or application within a system. It typically consists of a unique identifier (e.g., a username or password) and a set of permissions or privileges that the user is authorized to use."", ""needs_review"": true}",Pass/Fail,
4,What is Epoch?,False,"{""answer"": ""Epoch is a time unit used in computer science and physics to measure the duration of a period of time, typically used in the context of the AIE2026 course glossary."", ""needs_review"": true}",Pass/Fail,
5,I keep hearing about Epoch. What does it mean?,False,"{""answer"": ""Epoch is a time unit in the AIE2026 course glossary used to measure the duration of a specific event or process within a timeline."", ""needs_review"": true}",Pass/Fail,
6,What is Needs review?,False,"{""answer"": ""Needs review is a process where a project manager or team member reviews the work product of a project manager or team member to ensure that the work product is accurate and meets the project's requirements."", ""needs_review"": true}",Pass/Fail,
7,Define Needs review as it is used in this course.,False,"{""answer"": ""Needs review is a process where a project manager or team reviews the requirements, scope, and deliverables of a project to ensure they meet the project's objectives and constraints."", ""needs_review"": true}",Pass/Fail,
8,What is Rate limit?,False,"{""answer"": ""Rate limit is a security feature that restricts the number of requests a client can make to a service within a specific time period. This helps prevent attacks like DoS (Denial of Service) and ensures that only authorized users can access the service."", ""needs_review"": false}",Pass/Fail,
9,I keep hearing about Rate limit. What does it mean?,False,"{""answer"": ""Rate limit is a security feature that restricts the number of times a user or application can make a request to a service or API endpoint within a specific time frame. This helps prevent abuse and ensures that the service provider can handle the load."", ""needs_review"": true}",Pass/Fail,
